# 🧬 Dravya Labs Skin AI — Google Colab Training


This notebook trains an **EfficientNet-B0** model for skin disease classification.
It is **fully self-contained** — no file uploads or local imports required.

### Workflow
1. Set your `HF_TOKEN` in the Secrets sidebar (🔑 icon)
2. Run all cells — generates `skin_model.pth` + `model_config.json`
3. Model is saved to Google Drive for persistence
4. Download the model files and place them into your `skin/model/` folder locally

### Dataset


---
## 📋 Labeled Diseases (HAM10000 Dataset)

The **HAM10000** dataset classifies skin lesions into **7 categories**.
Below is what each label means and example images you should use for testing:

| # | Label (in dataset) | Full Name | Description | Test Image Tip |
|---|-------------------|-----------|-------------|----------------|
| 0 | **akiec** | Actinic Keratoses / Intraepithelial Carcinoma | Scaly, crusty patches on sun-exposed skin. Pre-cancerous. | Rough reddish/brown scaly patch on face/hands |
| 1 | **bcc** | Basal Cell Carcinoma | Most common skin cancer. Pearly/waxy bumps, often on face. | Shiny, translucent bump with visible blood vessels |
| 2 | **bkl** | Benign Keratosis (Solar Lentigo / Seborrheic Keratosis) | Non-cancerous growths. Brown, waxy, "stuck-on" appearance. | Brown, well-defined, waxy flat/raised lesion |
| 3 | **df** | Dermatofibroma | Hard, small, brownish-red bumps. Benign. | Small firm reddish-brown nodule on legs |
| 4 | **mel** | Melanoma | Most dangerous skin cancer. Irregular borders, multiple colors. | Asymmetric dark lesion with irregular edges |
| 5 | **nv** | Melanocytic Nevus (Mole) | Common benign moles. Round, uniform color. | Typical round, evenly-colored brown mole |
| 6 | **vasc** | Vascular Lesion (Angioma / Pyogenic Granuloma) | Blood vessel-related lesions. Red/purple spots. | Red/cherry-colored small round bump |

> **Note**: The model expects **dermoscopic images** (close-up skin photos, ideally taken with a dermatoscope). Regular photos work but accuracy may be lower.

### 🔍 What Input Images Should Look Like
- **Size**: Any size works (auto-resized to 224×224)
- **Format**: JPG, PNG, or any PIL-supported format
- **Content**: Close-up photo of a single skin lesion
- **Background**: Skin visible around the lesion


---


In [1]:
# Install missing packages (most are pre-installed in Colab)
!pip install -q scikit-learn tqdm matplotlib pillow requests datasets

# Verify GPU
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🖥️  Device: {device}")
if device.type == 'cuda':
    print(f"🚀 GPU: {torch.cuda.get_device_name(0)}")
    print(f"💾 VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
else:
    print("⚠️  No GPU detected! Training will happen on CPU (which is much slower).")



[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: C:\Users\u9780\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


🖥️  Device: cuda
🚀 GPU: NVIDIA GeForce RTX 4050 Laptop GPU
💾 VRAM: 6.0 GB


---
## 🔑 Step 2: Set HuggingFace Token

You need a HuggingFace token to download the dataset.

**Option A** (Recommended): Use Colab Secrets
1. Click the 🔑 key icon in the left sidebar
2. Add a secret named `HF_TOKEN` with your token
3. Toggle "Notebook access" ON



In [2]:
import os
from dotenv import load_dotenv

load_dotenv('.env.local')
HF_TOKEN = os.environ.get('HF_TOKEN')

if not HF_TOKEN:
    raise ValueError("❌ HF_TOKEN is empty! Make sure it is set in .env.local")

print(f"Token: {HF_TOKEN[:8]}...{HF_TOKEN[-4:]}")


Token: <HIDDEN>


---


In [3]:
import os
from pathlib import Path

LOCAL_MODEL_DIR = Path("model")
LOCAL_MODEL_DIR.mkdir(parents=True, exist_ok=True)

MODEL_SAVE_PATH = LOCAL_MODEL_DIR / "skin_model.pth"
CONFIG_SAVE_PATH = LOCAL_MODEL_DIR / "model_config.json"

print(f"📁 Local model dir: {LOCAL_MODEL_DIR.resolve()}")


📁 Local model dir: C:\Users\u9780\OneDrive\Documents\GitHub\Dravya-labs2\skin\model


---


In [4]:
# ========== CONFIGURATION ==========
# Adjust these values as needed

DATASET_NAME = "redlessone/Derm1M"   # Public 7-class skin lesion dataset
BATCH_SIZE = 32                      # Increase to 64 if you have A100
EPOCHS = 10                          # Increased epochs for better convergence
LEARNING_RATE = 1e-4
MAX_SAMPLES = 40000              # Support up to 20k samples (dataset permitting)
                                     # Set to None to use full dataset

print(f"📊 Dataset: {DATASET_NAME}")
print(f"📦 Batch size: {BATCH_SIZE}")
print(f"🔄 Epochs: {EPOCHS}")
print(f"📈 Learning rate: {LEARNING_RATE}")


📊 Dataset: Nagabu/HAM10000
📦 Batch size: 32
🔄 Epochs: 10
📈 Learning rate: 0.0001


---
## 📦 Step 5: Dataset Loader (Inlined)

This loads images directly from the HuggingFace Datasets Server API.


In [5]:
import logging
from PIL import Image
from torch.utils.data import Dataset
from datasets import load_dataset
import numpy as np

# Simple logger for Colab
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger('dravya_skin_ai')


class HFSkinDataset(Dataset):
    """
    PyTorch Dataset wrapper for Hugging Face `datasets` library.
    Efficiently handles large datasets (20k+) with caching directly to disk.
    """
    def __init__(self, dataset_name, split='train', transform=None, max_samples=None):
        self.transform = transform
        import huggingface_hub
        huggingface_hub.login(token=os.environ.get("HF_TOKEN"))
        logger.info(f"Loading dataset: {dataset_name} [{split}]...")
        
        # Load dataset stream or full 
        self.dataset = load_dataset(dataset_name, split=split)
        
        # Limit samples if requested
        if max_samples and max_samples < len(self.dataset):
             logger.info(f"Limiting to {max_samples} samples")
             self.dataset = self.dataset.select(range(max_samples))
        
        # Extract classes
        if 'label' in self.dataset.features:
            self.classes = self.dataset.features['label'].names
            self.class_to_idx = {name: i for i, name in enumerate(self.classes)}
        else:
            # Fallback if label names aren't in features
            unique_labels = sorted(list(set(self.dataset['label'])))
            self.classes = [str(l) for l in unique_labels]
            self.class_to_idx = {str(l): i for i, l in enumerate(unique_labels)}
            
        logger.info(f"Dataset ready: {len(self.dataset)} samples, {len(self.classes)} classes.")

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        item = self.dataset[idx]
        image = item['image'].convert('RGB')
        label = item['label']

        if self.transform:
            image = self.transform(image)

        return image, label


print("✅ HFSkinDataset class defined (using `datasets` library).")

✅ HFSkinDataset class defined (using `datasets` library).


---


In [6]:
import torch.nn as nn
from torchvision import transforms, models


def get_train_transforms():
    """Training transforms with ENHANCED augmentation."""
    return transforms.Compose([
        transforms.RandomResizedCrop(224, scale=(0.8, 1.0)),
        transforms.RandomHorizontalFlip(),
        transforms.RandomVerticalFlip(),
        transforms.RandomRotation(20),
        transforms.RandomAffine(degrees=0, translate=(0.1, 0.1)),
        transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406],
                             std=[0.229, 0.224, 0.225])
    ])


def get_val_transforms():
    """Validation transforms (no augmentation)."""
    return transforms.Compose([
        transforms.Resize(256),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406],
                             std=[0.229, 0.224, 0.225])
    ])


def get_skin_model(num_classes: int, device: torch.device, pretrained: bool = True) -> nn.Module:
    """Loads EfficientNet-B0 and replaces the classifier head."""
    weights = models.EfficientNet_B0_Weights.DEFAULT if pretrained else None
    model = models.efficientnet_b0(weights=weights)

    # Replace classifier: Sequential(Dropout, Linear(1280, 1000)) → Linear(1280, num_classes)
    in_features = model.classifier[1].in_features
    model.classifier[1] = nn.Linear(in_features, num_classes)

    model = model.to(device)
    return model


print("✅ Transforms & model builder defined.")

✅ Transforms & model builder defined.


---


In [7]:
print("⏳ Loading dataset via `datasets` library...\n")

full_dataset = HFSkinDataset(
    dataset_name=DATASET_NAME,
    split='train',
    max_samples=MAX_SAMPLES
)

classes = full_dataset.classes
num_classes = len(classes)
class_to_idx = full_dataset.class_to_idx

print(f"\n{'='*50}")
print(f"📊 Classes ({num_classes}):")
for i, cls in enumerate(classes):
    print(f"   {i}: {cls}")
print(f"📏 Total samples: {len(full_dataset)}")
print(f"{'='*50}")

2026-02-21 02:14:20,382 - INFO - Loading dataset: Nagabu/HAM10000 [train]...


⏳ Loading dataset via `datasets` library...



2026-02-21 02:14:21,032 - INFO - HTTP Request: HEAD https://huggingface.co/datasets/Nagabu/HAM10000/resolve/main/README.md "HTTP/1.1 307 Temporary Redirect"
2026-02-21 02:14:21,079 - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/datasets/Nagabu/HAM10000/5cb67dd5e0b4233eaca2e36b72fc21c9ad33cf9c/README.md "HTTP/1.1 200 OK"
2026-02-21 02:14:21,604 - INFO - HTTP Request: HEAD https://huggingface.co/datasets/Nagabu/HAM10000/resolve/5cb67dd5e0b4233eaca2e36b72fc21c9ad33cf9c/HAM10000.py "HTTP/1.1 404 Not Found"
2026-02-21 02:14:22,500 - INFO - HTTP Request: HEAD https://s3.amazonaws.com/datasets.huggingface.co/datasets/datasets/Nagabu/HAM10000/Nagabu/HAM10000.py "HTTP/1.1 404 Not Found"
2026-02-21 02:14:22,777 - INFO - HTTP Request: GET https://huggingface.co/api/datasets/Nagabu/HAM10000/revision/5cb67dd5e0b4233eaca2e36b72fc21c9ad33cf9c "HTTP/1.1 200 OK"
2026-02-21 02:14:23,199 - INFO - HTTP Request: HEAD https://huggingface.co/datasets/Nagabu/HAM10000/resolve/5cb67dd5e0b4


📊 Classes (2):
   0: melanoma
   1: nevus
📏 Total samples: 7818


---


In [8]:
import json

# Disease descriptions for the config
DISEASE_DESCRIPTIONS = {
    "akiec": "Actinic Keratoses and Intraepithelial Carcinoma (Bowen's disease) — pre-cancerous scaly patches",
    "bcc": "Basal Cell Carcinoma — most common skin cancer, pearly/waxy bumps",
    "bkl": "Benign Keratosis (Seborrheic Keratosis, Solar Lentigo, Lichen planus-like keratosis)",
    "df": "Dermatofibroma — benign hard brownish-red bumps, often on legs",
    "mel": "Melanoma — most dangerous skin cancer, irregular borders and colors",
    "nv": "Melanocytic Nevus (Mole) — common benign moles, usually round and uniform",
    "vasc": "Vascular Lesion (Angioma, Angiokeratoma, Pyogenic Granuloma, Hemorrhage)",
    "acne": "Acne — common skin condition characterized by red pimples (comedones/pustules) mainly on face and back",
    "eczema": "Eczema (Atopic Dermatitis) — chronic condition causing dry, red, itchy skin",
    "psoriasis": "Psoriasis — auto-immune condition causing scaly, red patches usually on knees/elbows",
    "rosacea": "Rosacea — long-term inflammatory skin condition causing reddened skin and visible blood vessels",
    "acne": "Acne — common skin condition characterized by red pimples (comedones/pustules) mainly on face and back",
    "eczema": "Eczema (Atopic Dermatitis) — chronic condition causing dry, red, itchy skin",
    "psoriasis": "Psoriasis — auto-immune condition causing scaly, red patches usually on knees/elbows",
    "rosacea": "Rosacea — long-term inflammatory skin condition causing reddened skin and visible blood vessels"
}

config_data = {
    "classes": classes,
    "class_to_idx": class_to_idx,
    "descriptions": {c: DISEASE_DESCRIPTIONS.get(c, f"Skin condition: {c}") for c in classes}
}

with open(CONFIG_SAVE_PATH, 'w') as f:
    json.dump(config_data, f, indent=4)

print(f"✅ Config saved to {CONFIG_SAVE_PATH}")
print(f"\n📋 Disease labels in config:")
for cls, desc in config_data['descriptions'].items():
    print(f"   • {cls}: {desc}")


✅ Config saved to model\model_config.json

📋 Disease labels in config:
   • melanoma: Skin condition: melanoma
   • nevus: Skin condition: nevus


---


In [9]:
from torch.utils.data import DataLoader, random_split

train_size = int(0.8 * len(full_dataset))
val_size = len(full_dataset) - train_size
train_subset, val_subset = random_split(full_dataset, [train_size, val_size])

# Disable base transform (applied via wrapper)
full_dataset.transform = None


class TransformSubset(torch.utils.data.Dataset):
    def __init__(self, subset, transform=None):
        self.subset = subset
        self.transform = transform
    def __getitem__(self, index):
        x, y = self.subset[index]
        if self.transform:
            x = self.transform(x)
        return x, y
    def __len__(self):
        return len(self.subset)


train_dataset = TransformSubset(train_subset, transform=get_train_transforms())
val_dataset = TransformSubset(val_subset, transform=get_val_transforms())

# num_workers=0 is safe on Colab
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)

print(f"✅ Train: {train_size} samples ({len(train_loader)} batches)")
print(f"✅ Val:   {val_size} samples ({len(val_loader)} batches)")

✅ Train: 6254 samples (196 batches)
✅ Val:   1564 samples (49 batches)


---


In [10]:
import torch.optim as optim

model = get_skin_model(num_classes=num_classes, device=device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"✅ EfficientNet-B0 ready on {device}")
print(f"📊 Total params: {total_params:,}")
print(f"📊 Trainable params: {trainable_params:,}")

✅ EfficientNet-B0 ready on cuda
📊 Total params: 4,010,110
📊 Trainable params: 4,010,110


---


In [ ]:
import copy
import time
from tqdm import tqdm
from torch.optim.lr_scheduler import ReduceLROnPlateau

best_model_wts = copy.deepcopy(model.state_dict())
best_acc = 0.0
patience = 5
early_stop_counter = 0
history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}

# Scheduler: Reduce LR if validation loss plateaus
scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.1, patience=3)

total_start = time.time()

for epoch in range(EPOCHS):
    print(f"\n{'='*50}")
    print(f"Epoch {epoch+1}/{EPOCHS}")
    print(f"{'='*50}")
    epoch_start = time.time()

    for phase in ['train', 'val']:
        if phase == 'train':
            model.train()
            dataloader = train_loader
        else:
            model.eval()
            dataloader = val_loader

        running_loss = 0.0
        running_corrects = 0

        pbar = tqdm(dataloader, desc=f"{phase.capitalize():>5}", leave=True)
        for inputs, labels in pbar:
            inputs = inputs.to(device)
            labels = labels.to(device)

            optimizer.zero_grad()

            with torch.set_grad_enabled(phase == 'train'):
                outputs = model(inputs)
                _, preds = torch.max(outputs, 1)
                loss = criterion(outputs, labels)

                if phase == 'train':
                    loss.backward()
                    optimizer.step()

            running_loss += loss.item() * inputs.size(0)
            running_corrects += torch.sum(preds == labels.data)
            pbar.set_postfix({'loss': f'{loss.item():.4f}'})

        epoch_loss = running_loss / len(dataloader.dataset)
        epoch_acc = running_corrects.double() / len(dataloader.dataset)

        print(f"  {phase:>5} → Loss: {epoch_loss:.4f}  Acc: {epoch_acc:.4f}")

        if phase == 'train':
            history['train_loss'].append(epoch_loss)
            history['train_acc'].append(epoch_acc.item())
        else:
            history['val_loss'].append(epoch_loss)
            history['val_acc'].append(epoch_acc.item())
            # Step scheduler based on validation loss
            scheduler.step(epoch_loss)

        if phase == 'val' and epoch_acc > best_acc:
            best_acc = epoch_acc
            best_model_wts = copy.deepcopy(model.state_dict())
            torch.save(best_model_wts, MODEL_SAVE_PATH)
            early_stop_counter = 0
            print(f"  🏆 New best model saved! (acc={epoch_acc:.4f})")
        elif phase == 'val':
            early_stop_counter += 1

    if early_stop_counter >= patience:
        print(f"\n⏹️  Early stopping after {epoch+1} epochs (no improvement for {patience} epochs).")
        break

    print(f"  ⏱️  Epoch time: {time.time() - epoch_start:.1f}s")

total_time = time.time() - total_start
print(f"\n{'='*50}")
print(f"🏁 Training Complete!")
print(f"   Total time: {total_time/60:.1f} minutes")
print(f"   Best Val Accuracy: {best_acc:.4f}")

# Save final best model
model.load_state_dict(best_model_wts)
torch.save(model.state_dict(), MODEL_SAVE_PATH)
print(f"   Model saved to: {MODEL_SAVE_PATH}")


Epoch 1/10


Train:   0%|          | 0/196 [00:00<?, ?it/s]

---


In [ ]:
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

epochs_range = range(1, len(history['train_loss']) + 1)

# Loss plot
ax1.plot(epochs_range, history['train_loss'], 'b-o', label='Train Loss', markersize=6)
ax1.plot(epochs_range, history['val_loss'], 'r-s', label='Val Loss', markersize=6)
ax1.set_xlabel('Epoch', fontsize=12)
ax1.set_ylabel('Loss', fontsize=12)
ax1.set_title('Training & Validation Loss', fontsize=14, fontweight='bold')
ax1.legend(fontsize=11)
ax1.grid(True, alpha=0.3)

# Accuracy plot
ax2.plot(epochs_range, history['train_acc'], 'b-o', label='Train Acc', markersize=6)
ax2.plot(epochs_range, history['val_acc'], 'r-s', label='Val Acc', markersize=6)
ax2.set_xlabel('Epoch', fontsize=12)
ax2.set_ylabel('Accuracy', fontsize=12)
ax2.set_title('Training & Validation Accuracy', fontsize=14, fontweight='bold')
ax2.legend(fontsize=11)
ax2.grid(True, alpha=0.3)
ax2.set_ylim(0, 1)

plt.tight_layout()
plt.savefig(LOCAL_MODEL_DIR / 'training_history.png', dpi=150, bbox_inches='tight')
plt.show()


---


In [ ]:
# Load saved model and verify it works
test_model = get_skin_model(num_classes=num_classes, device=device, pretrained=False)
state_dict = torch.load(MODEL_SAVE_PATH, map_location=device, weights_only=True)
test_model.load_state_dict(state_dict)
test_model.eval()

print(f"✅ Model loaded successfully from {MODEL_SAVE_PATH}")

# Test with a dummy image
dummy_img = Image.new('RGB', (224, 224), (128, 100, 80))
transform = get_val_transforms()
dummy_tensor = transform(dummy_img).unsqueeze(0).to(device)

with torch.no_grad():
    output = test_model(dummy_tensor)
    probs = torch.nn.functional.softmax(output, dim=1)
    top_prob, top_idx = torch.topk(probs, min(3, num_classes))

print("\n🔍 Test prediction (dummy image):")
for i in range(top_idx.shape[1]):
    idx = top_idx[0][i].item()
    prob = top_prob[0][i].item()
    label = classes[idx] if idx < len(classes) else 'Unknown'
    desc = DISEASE_DESCRIPTIONS.get(label, '')
    print(f"   {label} ({desc}): {prob:.4f}")

---
## 🧪 Step 14: Test with Your Own Image (Optional)



In [ ]:
import matplotlib.pyplot as plt
import io

print("Testing is recommended to be done via the FastAPI server locally.")


---


In [ ]:
print(f"\n✅ Done! The files are saved in your local skin/model/ folder:\n   skin/model/skin_model.pth\n   skin/model/model_config.json")
print(f"\nThen run locally: uvicorn app.main:app --reload")


---
## 📋 Disease Label Quick Reference

| Label | Disease | Severity | Example Input |
|-------|---------|----------|---------------|
| `akiec` | Actinic Keratosis | ⚠️ Pre-cancerous | Scaly red/brown patch |
| `bcc` | Basal Cell Carcinoma | 🔴 Cancerous | Pearly bump with blood vessels |
| `bkl` | Benign Keratosis | 🟢 Benign | Brown waxy "stuck-on" spot |
| `df` | Dermatofibroma | 🟢 Benign | Small firm reddish nodule |
| `mel` | Melanoma | 🔴 Dangerous | Irregular dark asymmetric lesion |
| `nv` | Melanocytic Nevus | 🟢 Benign | Round uniform brown mole |
